In [ ]:
import os
import json
import asyncio
import aiohttp
from PIL import Image
from io import BytesIO
from datasets import load_dataset
from torchvision import transforms

# ==============================
# CONFIG
# ==============================

SAVE_DIR = "laion_5m_64"
IMAGE_DIR = os.path.join(SAVE_DIR, "images")
CAPTION_FILE = os.path.join(SAVE_DIR, "captions.jsonl")

TARGET_IMAGES = 5_000_000
MAX_CONCURRENT = 256   # tune based on network
SIMILARITY_THRESHOLD = 0.28
IMAGE_SIZE = 64

os.makedirs(IMAGE_DIR, exist_ok=True)

# ==============================
# TRANSFORM
# ==============================

transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
])

# ==============================
# LOAD STREAMING DATASET
# ==============================

dataset = load_dataset(
    "laion/relaion400m",
    split="train",
    streaming=True
)

dataset = dataset.filter(
    lambda x: x["NSFW"] == "UNLIKELY"
    and x["similarity"] > SIMILARITY_THRESHOLD
    and len(x["caption"]) > 5
)

# ==============================
# ASYNC DOWNLOAD
# ==============================

semaphore = asyncio.Semaphore(MAX_CONCURRENT)

async def download_and_save(session, sample, index):
    async with semaphore:
        try:
            async with session.get(sample["url"], timeout=10) as response:
                if response.status != 200:
                    return None

                content = await response.read()
                image = Image.open(BytesIO(content)).convert("RGB")
                image = transform(image)

                filename = f"{index:010d}.jpg"
                filepath = os.path.join(IMAGE_DIR, filename)
                image.save(filepath, format="JPEG", quality=90)

                return {
                    "file": filename,
                    "caption": sample["caption"].strip()
                }

        except Exception:
            return None


async def main():
    count = 0
    tasks = []

    async with aiohttp.ClientSession() as session:
        for sample in dataset:
            if count >= TARGET_IMAGES:
                break

            task = asyncio.create_task(
                download_and_save(session, sample, count)
            )
            tasks.append(task)
            count += 1

            if len(tasks) >= MAX_CONCURRENT:
                results = await asyncio.gather(*tasks)
                tasks = []

                with open(CAPTION_FILE, "a") as f:
                    for r in results:
                        if r:
                            f.write(json.dumps(r) + "\n")

                print(f"Downloaded {count} images")

        # Final remaining tasks
        results = await asyncio.gather(*tasks)
        with open(CAPTION_FILE, "a") as f:
            for r in results:
                if r:
                    f.write(json.dumps(r) + "\n")

    print("Finished downloading 5M images.")


asyncio.run(main())